# Предсказание колоректального рака по микробиому

**Задача:** бинарная классификация (КРР / здоров) по профилю кишечных бактерий.

**Подход:** CLR-нормализация → отбор 46 таксонов → tuned XGBoost.

**Результат:** ROC AUC = **0.914** на валидации.

## Используемые библиотеки

- **pandas** — загрузка CSV, фильтрация, группировка, One-Hot Encoding (`get_dummies`), выравнивание колонок (`reindex`), сохранение результатов.
- **numpy** — числовые операции: логарифмы, массивы, `np.log1p`, `np.isnan`, статистика по вероятностям.
- **scikit-learn** — основной ML-стек:
  - `train_test_split` — стратифицированное разделение train/validation;
  - `StandardScaler` — масштабирование признаков для LogReg и RF;
  - `LogisticRegression`, `RandomForestClassifier` — baseline-модели;
  - `RandomizedSearchCV`, `StratifiedKFold` — подбор гиперпараметров и кросс-валидация;
  - `roc_auc_score`, `roc_curve`, `precision_recall_curve`, `average_precision_score` — метрики;
  - `confusion_matrix`, `ConfusionMatrixDisplay`, `classification_report` — оценка ошибок;
  - `permutation_importance` — честная оценка важности признаков.
- **xgboost** — градиентный бустинг (`XGBClassifier`). Финальная модель, показавшая лучший ROC AUC на валидации.
- **scikit-bio** — биоинформатический стек. Функция `clr` из `skbio.stats.composition` для центрированного логарифмического отношения — стандарт обработки композиционных микробиомных данных.
- **scipy.stats** — распределения `randint` и `uniform` для задания пространства поиска гиперпараметров в `RandomizedSearchCV`.
- **matplotlib** — визуализация: ROC-кривые, PR-кривые, bar chart метрик, confusion matrix.
- **re** — регулярные выражения для извлечения mOTU-номеров из имён колонок (`[ref_mOTU_v2_0777]` → `777`).

### Препроцессинг

- **CLR (Centered Log-Ratio)** — центрированное логарифмическое отношение. Переводит композиционные данные (доли, сумма = 1) в логарифмические отношения, убирая механическую зависимость между бактериями. Стандарт для микробиома.
- **Winsorize (1% / 99%)** — мягкое обрезание экстремумов по перцентилям. Не удаляет строки, но снижает влияние выбросов.
- **Импутация медианой** — заполнение пропусков медианой train. Устойчиво к скошенным распределениям abundanc'ов.
- **One-Hot Encoding** — кодирование категориальной переменной `cohort` (страна) в бинарные колонки. Val/test выровнены по колонкам train.
- **Label Encoding** — кодирование `gender` (`F`/`M`) в `0`/`1`.
- **log1p** — логарифм от `1 + x`. Сжимает длинный правый хвост `library_size`, сохраняя нули.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from skbio.stats.composition import clr

df = pd.read_csv("datasets/CU_hackathon_participants/train.csv")
df = df.dropna(subset=["label"]).reset_index(drop=True)
test_df = pd.read_csv("datasets/CU_hackathon_participants/test.csv")

train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)

gender_map = {"F": 0, "M": 1}
train_df["gender"] = train_df["gender"].map(gender_map)
val_df["gender"]   = val_df["gender"].map(gender_map)
test_df["gender"]  = test_df["gender"].map(gender_map)

train_df = pd.get_dummies(train_df, columns=["cohort"], prefix="cohort", dtype=int)
val_df   = pd.get_dummies(val_df,   columns=["cohort"], prefix="cohort", dtype=int)
test_df  = pd.get_dummies(test_df,  columns=["cohort"], prefix="cohort", dtype=int)

val_df  = val_df.reindex(columns=train_df.columns, fill_value=0)
test_df["label"] = 0
test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

meta_cols    = ["sample_id", "label"]
service_cols = ["age", "bmi", "library_size"]
bacteria_cols = [c for c in train_df.columns
                 if c not in meta_cols + service_cols + ["gender"]
                 and not c.startswith("cohort_")]

sums_train = train_df[bacteria_cols].sum(axis=1)
sums_val   = val_df[bacteria_cols].sum(axis=1)
sums_test  = test_df[bacteria_cols].sum(axis=1)

if (sums_train < 0.9).any() or (sums_train > 1.1).any():
    train_df[bacteria_cols] = train_df[bacteria_cols].div(sums_train.replace(0, 1), axis=0)
    val_df[bacteria_cols]   = val_df[bacteria_cols].div(sums_val.replace(0, 1), axis=0)
    test_df[bacteria_cols]  = test_df[bacteria_cols].div(sums_test.replace(0, 1), axis=0)

lo = train_df[bacteria_cols + service_cols].quantile(0.01)
hi = train_df[bacteria_cols + service_cols].quantile(0.99)
for col in bacteria_cols + service_cols:
    train_df[col] = train_df[col].clip(lo[col], hi[col])
    val_df[col]   = val_df[col].clip(lo[col], hi[col])
    test_df[col]  = test_df[col].clip(lo[col], hi[col])

num_cols = bacteria_cols + service_cols
medians  = train_df[num_cols].median()
train_df[num_cols] = train_df[num_cols].fillna(medians).fillna(0)
val_df[num_cols]   = val_df[num_cols].fillna(medians).fillna(0)
test_df[num_cols]  = test_df[num_cols].fillna(medians).fillna(0)

train_bac = train_df[bacteria_cols].to_numpy() + 1
val_bac   = val_df[bacteria_cols].to_numpy() + 1
test_bac  = test_df[bacteria_cols].to_numpy() + 1

train_df[bacteria_cols] = clr(train_bac)
val_df[bacteria_cols]   = clr(val_bac)
test_df[bacteria_cols]  = clr(test_bac)

train_df["library_size"] = np.log1p(train_df["library_size"])
val_df["library_size"]   = np.log1p(val_df["library_size"])
test_df["library_size"]  = np.log1p(test_df["library_size"])

train_df.to_csv("modified_train.csv", index=False)
val_df.to_csv("modified_validation.csv", index=False)
test_df.drop(columns=["label"]).to_csv("modified_test.csv", index=False)

## Модель

Отбираем 46 таксонов из литературы и предыдущих анализов, обучаем XGBoost с подбором гиперпараметров через `RandomizedSearchCV` (5-fold CV).

In [2]:
import re
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import randint, uniform
from xgboost import XGBClassifier

train = pd.read_csv("modified_train.csv")
val   = pd.read_csv("modified_validation.csv")
test  = pd.read_csv("modified_test.csv")

motu_to_col = {}
for c in train.columns:
    m = re.search(r"_(\d+)\]\s*$", c)
    if m:
        motu_to_col[int(m.group(1))] = c

def cols_from_motus(motus):
    seen, out = set(), []
    for m in motus:
        if m in seen:
            continue
        seen.add(m)
        col = motu_to_col.get(m)
        if col and col not in out:
            out.append(col)
    return out

MY_CUSTOM_MOTUS = [
    5867, 7656, 515, 4616, 4961, 5431, 2101, 1403, 754, 2102,
    276, 777, 6490, 7777, 5554, 6257, 1404, 1517, 429, 148,
    5486, 6156, 5245, 7596, 6224, 4636, 520, 5989, 6788, 1309,
    1074, 1377, 886, 1475, 6105, 7553, 754, 776, 777, 1403,
    4614, 429, 1376, 276, 515, 4513, 882, 1145, 4961, 1517,
    2101, 2102, 4616, 531, 4738, 6835, 5867, 5742, 5431, 6490, 7656,
]
custom_cols = cols_from_motus(MY_CUSTOM_MOTUS)

def sanitize(df, cols):
    cols = list(dict.fromkeys(cols))
    mapping = {c: f"f{i}" for i, c in enumerate(cols)}
    return df[cols].rename(columns=mapping)

X_tr = sanitize(train, custom_cols)
X_va = sanitize(val,   custom_cols)
X_te = sanitize(test,  custom_cols)

pos_weight = (train["label"] == 0).sum() / max((train["label"] == 1).sum(), 1)

param_dist = {
    "n_estimators":     [300, 500, 800],
    "max_depth":        randint(3, 7),
    "learning_rate":    uniform(0.02, 0.13),
    "subsample":        uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "reg_lambda":       [1, 3, 5, 10],
}

base = XGBClassifier(
    scale_pos_weight=pos_weight,
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    base, param_dist,
    n_iter=30, cv=cv,
    scoring="roc_auc",
    random_state=42, n_jobs=-1,
    error_score="raise",
)
search.fit(X_tr, train["label"])

best_model = search.best_estimator_
proba_val = best_model.predict_proba(X_va)[:, 1]
auc_val = roc_auc_score(val["label"], proba_val)
print(f"CV AUC:  {search.best_score_:.4f}")
print(f"Val AUC: {auc_val:.4f}")

CV AUC:  0.8314
Val AUC: 0.9030


## Сравнение моделей и ROC-кривые

LogReg (L1), RandomForest и XGBoost на одних и тех же 46 признаках.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, roc_auc_score

models = {
    "LogReg (L1)": make_pipeline(
        StandardScaler(),
        LogisticRegression(penalty="l1", solver="liblinear", C=0.1,
                           max_iter=2000, class_weight="balanced"),
    ),
    "RandomForest": make_pipeline(
        StandardScaler(),
        RandomForestClassifier(n_estimators=500, max_depth=10,
                               min_samples_leaf=3, class_weight="balanced",
                               random_state=42, n_jobs=-1),
    ),
    "XGBoost (tuned)": best_model,
}

results = {}
for name, model in models.items():
    Xv = X_va if name.startswith("XGBoost") else val[custom_cols]
    Xt = X_tr if name.startswith("XGBoost") else train[custom_cols]
    model.fit(Xt, train["label"])
    proba = model.predict_proba(Xv)[:, 1]
    fpr, tpr, _ = roc_curve(val["label"], proba)
    auc = roc_auc_score(val["label"], proba)
    results[name] = {"fpr": fpr, "tpr": tpr, "auc": auc}
    print(f"{name:20s}: AUC = {auc:.4f}")

plt.figure(figsize=(8, 7))
colors = {"LogReg (L1)": "#1f77b4",
          "RandomForest": "#2ca02c",
          "XGBoost (tuned)": "#d62728"}
for name, r in results.items():
    plt.plot(r["fpr"], r["tpr"], lw=2, color=colors[name],
             label=f"{name} (AUC = {r['auc']:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", lw=1.5,
         label="Случайное угадывание (AUC = 0.500)")
plt.xlim([0, 1]); plt.ylim([0, 1.05])
plt.xlabel("Доля ложноположительных результатов (FPR)")
plt.ylabel("Доля верно выбранных случаев (TPR)")
plt.title("ROC-кривые моделей на валидации")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("roc_curves_validation.png", dpi=150)
plt.show()

## Precision-Recall и сравнение метрик

Для медицинского скрининга важнее не ROC-AUC, а баланс recall/precision.

In [ ]:
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    accuracy_score, precision_score, recall_score, f1_score,
)

plt.figure(figsize=(8, 7))
for name, r in results.items():
    proba = models[name].predict_proba(
        X_va if name.startswith("XGBoost") else val[custom_cols]
    )[:, 1]
    precision, recall, _ = precision_recall_curve(val["label"], proba)
    ap = average_precision_score(val["label"], proba)
    plt.plot(recall, precision, lw=2, color=colors[name],
             label=f"{name} (PR-AUC = {ap:.3f})")
plt.axhline(y=val["label"].mean(), color="gray", linestyle="--", lw=1.5,
            label=f"Случайное (Precision = {val['label'].mean():.2f})")
plt.xlim([0, 1]); plt.ylim([0, 1.05])
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall кривые на валидации")
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("pr_curves_validation.png", dpi=150)
plt.show()

def metrics_row(y_true, proba):
    pred = (proba >= 0.5).astype(int)
    return {
        "Accuracy":  accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall":    recall_score(y_true, pred, zero_division=0),
        "F1":        f1_score(y_true, pred, zero_division=0),
        "ROC-AUC":   roc_auc_score(y_true, proba),
        "PR-AUC":    average_precision_score(y_true, proba),
    }

rows = {}
for name in models:
    proba = models[name].predict_proba(
        X_va if name.startswith("XGBoost") else val[custom_cols]
    )[:, 1]
    rows[name] = metrics_row(val["label"], proba)

df_metrics = pd.DataFrame(rows).T.round(4)
print(df_metrics.to_string())

metric_names = df_metrics.columns.tolist()
x = np.arange(len(metric_names))
width = 0.25
fig, ax = plt.subplots(figsize=(11, 6))
for i, name in enumerate(df_metrics.index):
    vals = df_metrics.loc[name, metric_names].values
    offset = (i - 1) * width
    bars = ax.bar(x + offset, vals, width, label=name,
                  color=colors[name], edgecolor="black", linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01,
                f"{v:.2f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(metric_names)
ax.set_ylim(0, 1.10); ax.set_ylabel("Значение метрики")
ax.set_title("Сравнение моделей по метрикам (validation)")
ax.legend(loc="upper right", ncol=3)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("metrics_comparison.png", dpi=150)
plt.show()

## Confusion Matrix XGBoost

Смотрим на FN (пропущенные больные) — самое критичное для скрининга.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

proba_val = best_model.predict_proba(X_va)[:, 1]
y_pred = (proba_val >= 0.5).astype(int)
cm = confusion_matrix(val["label"], y_pred)
tn, fp, fn, tp = cm.ravel()

print(classification_report(val["label"], y_pred, target_names=["Здоров", "КРР"]))
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=["Здоров", "КРР"]).plot(
    ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title(f"Confusion Matrix — XGBoost\n"
             f"Recall={recall_score(val['label'], y_pred):.3f}, "
             f"Precision={precision_score(val['label'], y_pred):.3f}")
plt.tight_layout()
plt.savefig("confusion_matrix_xgb.png", dpi=150)
plt.show()

## Сабмит

Формат: `sample_id` + `label` (вероятность КРР, не 0/1).

In [ ]:
proba_test = best_model.predict_proba(X_te)[:, 1]
submission = pd.DataFrame({
    "sample_id": test["sample_id"],
    "label":     proba_test,
})
submission.to_csv("submission.csv", index=False)
print(f"Сохранено: submission.csv ({len(submission)} строк)")
print(submission["label"].describe().round(4))